In [ ]:
!pip install requests beautifulsoup4 tqdm urllib3 sentence-transformers faiss-cpu pypdf PyPDF2 transformers sentencepiece torch accelerate bitsandbytes spacy networkx pyvis duckduckgo_search streamlit pyngrok streamlit-option-menu textstat pyjwt bcrypt wikipedia pandas plotly matplotlib wordcloud -q
!python -m spacy download en_core_web_sm -q
print("All packages installed!")

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

APP_DIR = "/content/drive/MyDrive/PolicyNav"
os.makedirs(APP_DIR, exist_ok=True)
os.makedirs(os.path.join(APP_DIR, "documents"), exist_ok=True)
os.makedirs(os.path.join(APP_DIR, "graphs"), exist_ok=True)
os.environ["APP_DIR"] = APP_DIR
print(f"APP_DIR = {APP_DIR}")
print(f"Upload PDFs to: {APP_DIR}/documents/")

In [ ]:
import sqlite3, os
base_dir = os.path.join(os.environ.get("APP_DIR","/content/drive/MyDrive/PolicyNav"),"documents")
db_path  = os.path.join(os.environ.get("APP_DIR","/content/drive/MyDrive/PolicyNav"),"policies_meta.db")
os.makedirs(base_dir, exist_ok=True)
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA journal_mode=WAL;")
conn.execute("""CREATE TABLE IF NOT EXISTS policies
               (id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT, url TEXT UNIQUE, local_path TEXT)""")
conn.commit(); conn.close()
print("DB initialized. PDF folder:", base_dir)

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm
import urllib3
import time
import os
import sqlite3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

# Paths
base_dir = "/content/drive/MyDrive/PolicyNav/documents"
db_path = "/content/drive/MyDrive/PolicyNav/policies_meta.db"

os.makedirs(base_dir, exist_ok=True)

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

# -----------------------------
# Known Official Policy PDFs
# -----------------------------
# -----------------------------
# Known Official Policy PDFs
# -----------------------------
direct_pdfs = [
    # -- Your Original Scrapes --
    ("NEP","https://www.education.gov.in/sites/upload_files/mhrd/files/NEP_Final_English_0.pdf"),
    ("PMJDY","https://www.pmjdy.gov.in/files/E-Documents/PMJDY_Brochure_ENG.pdf"),
    ("PMAY_Urban","https://pmay-urban.gov.in/uploads/guidelines/Operational-Guidelines-of-PMAY-U.pdf"),
    ("PMAY_Gramin","https://pmayg.dord.gov.in/netiayHome/Uploaded/Guidelines-English_Book_Final.pdf"),
    ("PMFBY","https://pmfby.gov.in/pdf/Revamped%20Operational%20Guidelines_17th%20August%202020.pdf"),
    ("PMKISAN","https://pmkisan.gov.in/Documents/RevisedPM-KISANOperationalGuidelines(English).pdf"),
    ("JalJeevanMission","https://jaljeevanmission.gov.in/sites/default/files/guideline/JJM_Operational_Guidelines.pdf"),
    ("SHC","https://www.agriwelfare.gov.in/sites/default/files/Guidelines_for_Demonstrations_under_SHC_Scheme.pdf"),
    ("SwachhBharat","https://archive.ids.ac.uk/clts/sites/communityledtotalsanitation.org/files/SwachhBharatMissionGraminGuidelines.pdf"),
    ("PMKVY","https://www.msde.gov.in/static/uploads/2024/02/PMKVY-4.0-Guidelines_final-copy.pdf"),
    ("DigitalIndia","https://www.meity.gov.in/static/uploads/2024/03/Brochure.pdf"),
    ("SmartCities","https://mohua.gov.in/dataSmartCities/uploads/resource/resourceDoc/Resource_Doc_1723187622_Making_a_City_Smart_Learnings_from_the_Smart_Cities_Mission.pdf"),

    # -- NEW ADDITIONS: Healthcare & Nutrition --
    ("AyushmanBharat_PMJAY", "https://pmjay.gov.in/sites/default/files/2018-07/Guidelines_on_Process_of_Empanelment_of_Hospitals_0.pdf"),
    ("Poshan_Abhiyaan", "https://icds-wcd.nic.in/nnm/NNM-Web-Contents/UPPER-MENU/AboutNNM/PM_Overarching_Scheme_Guidelines.pdf"),

    # -- NEW ADDITIONS: Employment & Rural --
    ("MGNREGA", "https://nrega.nic.in/netnrega/writereaddata/Circulars/MGNREGA_Operational_Guidelines_2013_eng.pdf"),
    ("AMRUT_Mission", "https://mohua.gov.in/upload/uploadfiles/files/AMRUTGuidelines.pdf"),

    # -- NEW ADDITIONS: Infrastructure & Energy --
    ("PM_Ujjwala_Yojana", "https://www.pmuy.gov.in/documents/Ujjwala-Brochure.pdf"),
    ("Saubhagya_Electricity", "https://saubhagya.gov.in/documents/Guidelines_Saubhagya.pdf"),

    # -- NEW ADDITIONS: Finance & Pensions --
    ("Atal_Pension_Yojana", "https://npscra.nsdl.co.in/nsdl/scheme-details/APY_Scheme_Details.pdf"),
    ("StandUp_India", "https://www.standupmitra.in/Content/StandupIndia/SUISchemeGuidelines.pdf")
]


# -----------------------------
# Policy Landing Pages
# -----------------------------
policy_pages = {
    # -- Your Original Scrapes --
    "PMJDY":"https://pmjdy.gov.in/",
    "PMAY_Urban":"https://pmay-urban.gov.in/",
    "JalJeevanMission":"https://jaljeevanmission.gov.in/",
    "DigitalIndia":"https://digitalindia.gov.in/",
    "SkillIndia":"https://www.msde.gov.in/",

    # -- NEW ADDITIONS --
    "AyushmanBharat": "https://pmjay.gov.in/",
    "StartupIndia": "https://www.startupindia.gov.in/",
    "MakeInIndia": "https://www.makeinindia.com/",
    "WomenAndChildDev": "https://wcd.nic.in/",           # Pulls Beti Bachao Beti Padhao, etc.
    "RuralDevelopment": "https://rural.nic.in/",         # Pulls Rural infrastructure schemes
    "MudraYojana": "https://www.mudra.org.in/",
    "PM_SVANidhi": "https://pmsvanidhi.mohua.gov.in/",   # Street vendor loans
    "MinistryOfHealth": "https://main.mohfw.gov.in/"
}


# -----------------------------
# Find PDFs from websites
# -----------------------------
def find_pdfs(url):

    try:
        r = requests.get(url,headers=HEADERS,timeout=20,verify=False)

        soup = BeautifulSoup(r.text,"html.parser")

        pdfs = []

        for link in soup.find_all("a",href=True):

            if ".pdf" in link["href"].lower():

                pdfs.append(urljoin(url,link["href"]))

        return pdfs

    except:

        return []


# -----------------------------
# Check already downloaded
# -----------------------------
def already_downloaded(url):

    conn = sqlite3.connect(db_path)
    c = conn.cursor()

    c.execute("SELECT id FROM policies WHERE url=?",(url,))
    data = c.fetchone()

    conn.close()

    return data is not None


# -----------------------------
# Download PDF (CORRECTED)
# -----------------------------
def download_pdf(url,scheme):
    filename = url.split("/")[-1].split("?")[0]
    save_name = f"{scheme}_{filename}"
    path = os.path.join(base_dir,save_name)

    # If it exists, skip instantly!
    if os.path.exists(path) or already_downloaded(url):
        return False # Returns False meaning "Did not download"

    try:
        r = requests.get(url,headers=HEADERS,stream=True,timeout=30,verify=False)
        if "application/pdf" not in r.headers.get("Content-Type","").lower() and not r.content.startswith(b"%PDF"):
            return False

        with open(path,"wb") as f:
            for chunk in r.iter_content(1024):
                f.write(chunk)

        conn = sqlite3.connect(db_path)
        c = conn.cursor()
        c.execute("INSERT INTO policies (title,url,local_path) VALUES (?,?,?)", (scheme,url,path))
        conn.commit()
        conn.close()

        print("Downloaded:",save_name)
        return True # Returns True meaning "Successfully downloaded"

    except:
        print("Failed:",url)
        return False

# -----------------------------
# Combine sources & Download (CORRECTED)
# -----------------------------
pdf_mapping = []

# Add direct policy PDFs
for scheme,url in direct_pdfs:
    pdf_mapping.append((url,scheme))

print("Scanning policy websites...")
for scheme,url in policy_pages.items():
    pdfs = find_pdfs(url)
    print(f"{scheme}: {len(pdfs)} PDFs found")
    for pdf in pdfs:
        pdf_mapping.append((pdf,scheme))

print("\nTotal PDFs found:",len(pdf_mapping))
print("Checking Drive for new files...\n")

for pdf_url,scheme in pdf_mapping:
    # Only trigger the 1-second pause if a brand new file was actually downloaded
    was_downloaded = download_pdf(pdf_url,scheme)
    if was_downloaded:
        time.sleep(1)


# -----------------------------
# Final Count
# -----------------------------
conn = sqlite3.connect(db_path)
c = conn.cursor()

c.execute("SELECT COUNT(*) FROM policies")
count = c.fetchone()[0]

conn.close()

print("\nDownload complete!")
print("Total PDFs stored in database:",count)

In [ ]:
from pypdf import PdfReader
import os

base_dir  = os.path.join(os.environ.get("APP_DIR","/content/drive/MyDrive/PolicyNav"),"documents")
documents = []

for file in os.listdir(base_dir):
    if file.endswith(".pdf"):
        path = os.path.join(base_dir, file)
        try:
            reader = PdfReader(path)
            text = "\n".join(p.extract_text() or "" for p in reader.pages)
            if text.strip():
                documents.append({"file": file, "text": text})
        except Exception as e:
            print("Error reading:", file, e)

print("Total PDFs processed:", len(documents))

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss, pickle, numpy as np, os

def split_text(text, chunk_size=500):
    words = text.split()
    return [' '.join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

all_chunks = []
for doc in documents:
    for chunk in split_text(doc['text']):
        all_chunks.append({"source": doc["file"], "text": chunk})
print("Total chunks:", len(all_chunks))

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [c['text'] for c in all_chunks]
embeddings = embed_model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)

dim   = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.array(embeddings))
print("Vectors stored:", index.ntotal)

APP_DIR = os.environ.get("APP_DIR","/content/drive/MyDrive/PolicyNav")
faiss.write_index(index, os.path.join(APP_DIR,"policy_vector_db.index"))
with open(os.path.join(APP_DIR,"chunks.pkl"),"wb") as f:
    pickle.dump(all_chunks, f)
print("Vector DB saved!")

In [ ]:
%%writefile knowledge_graph.py
import os
import spacy
from pyvis.network import Network

try:
    nlp_kg = spacy.load("en_core_web_lg")
except Exception:
    try:
        nlp_kg = spacy.load("en_core_web_md")
    except Exception:
        try:
            nlp_kg = spacy.load("en_core_web_sm")
        except Exception:
            nlp_kg = None

def generate_knowledge_graph(chunks_subset, nlp=None):
    if nlp is None:
        nlp = nlp_kg
    if nlp is None:
        raise RuntimeError("No spaCy model available. Run: python -m spacy download en_core_web_sm")

    net = Network(height="500px", width="100%", bgcolor="#131314", font_color="#e3e3e3", notebook=False)
    seen_nodes = set()
    seen_edges = set()

    for chunk in chunks_subset:
        filename = chunk["source"]
        doc = nlp(chunk["text"])

        if filename not in seen_nodes:
            net.add_node(filename, label=filename, color="#a8c7fa", size=20)
            seen_nodes.add(filename)

        for ent in doc.ents:
            if ent.label_ in ["ORG", "GPE", "DATE"]:
                ent_text = ent.text.strip()
                if not ent_text or ent_text.isdigit() or len(ent_text) < 2:
                    continue
                if ent_text not in seen_nodes:
                    net.add_node(ent_text, label=ent_text, color="#c4c7c5", size=12)
                    seen_nodes.add(ent_text)
                edge_key = (filename, ent_text)
                if edge_key not in seen_edges:
                    net.add_edge(filename, ent_text, title="mentions")
                    seen_edges.add(edge_key)

    net.set_options("""
    {
      "physics": {
        "enabled": true,
        "stabilization": { "enabled": true, "iterations": 300, "fit": true },
        "barnesHut": {
          "gravitationalConstant": -8000,
          "centralGravity": 0.3,
          "springLength": 120,
          "springConstant": 0.04,
          "damping": 0.09
        }
      },
      "interaction": {
        "hover": true,
        "navigationButtons": true,
        "keyboard": true
      }
    }
    """)

    APP_DIR = os.environ.get("APP_DIR", "/content/drive/MyDrive/PolicyNav")
    graph_path = os.path.join(APP_DIR, "policy_graph.html")
    net.save_graph(graph_path)
    return graph_path

print("knowledge_graph.py created!")

In [ ]:
%%writefile readability_utils.py
import textstat

class ReadabilityAnalyzer:
    def __init__(self, text):
        self.text        = text
        self.num_sentences = max(1, text.count('.'))
        self.num_words   = len(text.split())
        self.char_count  = len(text)
        self.syllables   = textstat.syllable_count(text)
        self.complex_words = textstat.lexicon_count(text, removepunct=True)

    def get_all_metrics(self):
        return {
            "Flesch Reading Ease":  textstat.flesch_reading_ease(self.text),
            "Flesch-Kincaid Grade": textstat.flesch_kincaid_grade(self.text),
            "Gunning Fog":          textstat.gunning_fog(self.text),
            "SMOG Index":           textstat.smog_index(self.text),
            "Coleman-Liau":         textstat.coleman_liau_index(self.text),
        }
print("readability_utils.py created!")

In [ ]:
%%writefile styles.py
CSS = """
<style>
    @import url('https://fonts.googleapis.com/css2?family=Inter:wght@300;400;500;600&family=DM+Mono:wght@400;500&display=swap');

    /* ===================== GLOBAL ===================== */
    .stApp, p, h1, h2, h3, h4, h5, h6, span, div, input, button {
        font-family: 'Inter', sans-serif;
    }

    span.material-symbols-rounded,
    i.material-icons,
    [data-testid="collapsedControl"] span,
    [data-testid="baseButton-header"] span {
        font-family: 'Material Symbols Rounded', 'Material Icons', sans-serif !important;
    }

    /* Gemini-style Dark Backgrounds */
    .stApp { background: #131314 !important; }
    .main  { background: #131314 !important; }

    #MainMenu { visibility: hidden; }
    footer    { visibility: hidden; }
    header { background: transparent !important; }

    /* ===================== SIDEBAR ===================== */
    section[data-testid="stSidebar"] {
        background-color: #1e1f20 !important;
        border-right: 1px solid #444746 !important;
    }

    section[data-testid="stSidebar"] .stScrollableContainer {
        padding: 1.5rem 0.8rem !important;
    }

    button[data-testid="collapsedControl"] {
        color: #c4c7c5 !important;
        background-color: transparent !important;
        border: none !important;
        transition: background 0.2s ease !important;
        padding: 0.5rem !important;
    }

    button[data-testid="collapsedControl"]:hover {
        background-color: #282a2c !important;
        border-radius: 8px !important;
    }

    .sidebar-spacer { min-height: 45vh; }

    .menu-label {
        color: #8e918f;
        font-size: 0.75rem;
        font-weight: 600;
        padding: 0.5rem 0.8rem;
        margin-top: 0.5rem;
        letter-spacing: 0.05em;
        text-transform: uppercase;
    }

    /* Sidebar buttons */
    section[data-testid="stSidebar"] .stButton > button {
        background: transparent !important;
        color: #c4c7c5 !important;
        font-size: 0.9rem !important;
        padding: 0.65rem 0.8rem !important;
        font-weight: 400 !important;
        border-radius: 8px !important;
        border: none !important;
        box-shadow: none !important;
        margin: 2px 0 !important;
        text-align: left !important;
        display: flex !important;
        justify-content: flex-start !important;
        transition: background 0.15s ease, color 0.15s ease !important;
    }

    section[data-testid="stSidebar"] .stButton > button:hover {
        background: #282a2c !important;
        color: #e3e3e3 !important;
    }

    /* Selected state */
    section[data-testid="stSidebar"] .stButton > button[kind="primary"] {
        background: #004a77 !important;
        color: #c2e7ff !important;
        font-weight: 500 !important;
    }

    /* Clean Sidebar User Profile block */
    .sidebar-bottom-profile {
        display: flex;
        align-items: center;
        padding: 12px 14px;
        border-radius: 8px;
        cursor: pointer;
        transition: background 0.2s;
        border-top: 1px solid #444746;
        margin-top: 0.5rem;
    }

    .sidebar-bottom-profile:hover {
        background: #282a2c;
    }

    .profile-name {
        color: #e3e3e3;
        font-size: 0.95rem;
        font-weight: 500;
        white-space: nowrap;
        text-overflow: ellipsis;
        overflow: hidden;
        display: flex;
        align-items: center;
    }

    .main .block-container {
        padding: 2.5rem 3rem !important;
        max-width: 100% !important;
    }

    /* ===================== LOGO ===================== */
    .logo-container {
        text-align: center;
        margin-bottom: 1.5rem;
        padding-bottom: 1rem;
    }

    .logo-icon {
        display: inline-flex;
        padding: 0.7rem;
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        margin-bottom: 0.8rem;
    }

    .logo-icon svg {
        width: 2rem;
        height: 2rem;
        color: #a8c7fa;
    }

    .logo-text {
        font-size: 1.4rem;
        font-weight: 600;
        color: #e3e3e3;
        letter-spacing: -0.02em;
        margin: 0;
    }

    .logo-subtext {
        color: #8e918f;
        font-size: 0.75rem;
        letter-spacing: 0.05em;
        text-transform: uppercase;
        margin: 0;
    }

    /* ===================== PAGE TITLE ===================== */
    .page-title {
        font-size: 1.8rem;
        font-weight: 500;
        color: #e3e3e3;
        margin-bottom: 0.3rem;
        letter-spacing: -0.03em;
    }

    .page-subtitle {
        color: #8e918f;
        font-size: 0.9rem;
        margin-bottom: 2rem;
    }

    /* ===================== INPUTS ===================== */
    .stTextInput > label,
    .stSelectbox > label {
        color: #c4c7c5 !important;
        font-size: 0.8rem !important;
        font-weight: 500 !important;
        margin-bottom: 0.4rem !important;
    }

    .stTextInput > div > div > input {
        background: #1e1f20 !important;
        border: 1px solid #444746 !important;
        border-radius: 8px !important;
        padding: 0.7rem 1rem !important;
        color: #e3e3e3 !important;
        font-size: 0.9rem !important;
        transition: all 0.2s ease !important;
    }

    .stTextInput > div > div > input:focus {
        border-color: #a8c7fa !important;
        box-shadow: none !important;
        background: #282a2c !important;
    }

    /* ===================== BUTTONS ===================== */
    .main .stButton > button {
        background: #a8c7fa !important;
        color: #062e6f !important;
        border: none !important;
        border-radius: 8px !important;
        padding: 0.65rem 1rem !important;
        font-size: 0.9rem !important;
        font-weight: 500 !important;
        width: 100% !important;
        margin: 0.3rem 0 !important;
        transition: opacity 0.2s ease !important;
        box-shadow: none !important;
    }

    .main .stButton > button:hover {
        opacity: 0.9 !important;
        transform: translateY(-1px) !important;
    }

    .main .stButton > button[kind="secondary"] {
        background: #1e1f20 !important;
        color: #a8c7fa !important;
        border: 1px solid #444746 !important;
    }

    .main .stButton > button[kind="secondary"]:hover {
        background: #282a2c !important;
        border-color: #a8c7fa !important;
        transform: none !important;
    }

    /* ===================== ALERTS ===================== */
    .stAlert {
        background: #1e1f20 !important;
        border: 1px solid #444746 !important;
        border-radius: 8px !important;
        color: #e3e3e3 !important;
    }

    /* ===================== ADMIN BADGE ===================== */
    .admin-badge {
        background: #004a77;
        color: #c2e7ff;
        padding: 4px 12px;
        border-radius: 20px;
        font-size: 0.65rem;
        font-weight: 600;
        letter-spacing: 0.08em;
        text-transform: uppercase;
        display: inline-block;
    }

    /* ===================== DASHBOARD STAT CARDS ===================== */
    .stat-card {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.8rem 1.5rem;
        text-align: center;
        transition: all 0.2s ease;
        position: relative;
        overflow: hidden;
    }

    .stat-number {
        font-size: 2.2rem;
        font-weight: 500;
        color: #e3e3e3;
        margin: 0;
        letter-spacing: -0.04em;
        font-family: 'DM Mono', monospace !important;
    }

    .stat-label {
        color: #8e918f;
        font-size: 0.75rem;
        margin: 0.3rem 0 0 0;
        text-transform: uppercase;
        letter-spacing: 0.05em;
        font-weight: 500;
    }

    /* ===================== READABILITY METRIC CARDS ===================== */
    .metric-card {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.4rem;
        position: relative;
    }

    .metric-title {
        color: #8e918f;
        font-size: 0.78rem;
        font-weight: 600;
        text-transform: uppercase;
        margin-bottom: 0.3rem;
        letter-spacing: 0.05em;
    }

    .metric-value {
        font-size: 2rem;
        font-weight: 500;
        color: #e3e3e3;
        font-family: 'DM Mono', monospace !important;
        margin: 0.5rem 0 0.8rem 0;
    }

    .metric-bar-track {
        height: 4px;
        background: #282a2c;
        border-radius: 4px;
        overflow: hidden;
        margin-bottom: 0.5rem;
    }

    .metric-bar-fill {
        height: 100%;
        border-radius: 4px;
        transition: width 1s ease;
    }

    .metric-interpretation {
        font-size: 0.75rem;
        color: #c4c7c5;
        margin: 0;
    }

    /* Level Banner */
    .level-banner {
        background: #1e1f20;
        border: 1px solid #444746;
        border-radius: 12px;
        padding: 1.8rem 2rem;
        display: flex;
        align-items: center;
        gap: 1.5rem;
        margin-bottom: 1.5rem;
    }

    .level-icon {
        width: 56px; height: 56px;
        border-radius: 12px;
        display: flex; align-items: center; justify-content: center;
        font-size: 1.6rem;
    }

    .level-title { font-size: 1.3rem; font-weight: 500; color: #e3e3e3; margin: 0 0 0.2rem 0; }
    .level-desc { font-size: 0.82rem; color: #8e918f; margin: 0; }
    .level-grade { margin-left: auto; text-align: center; }
    .level-grade-num { font-size: 2.4rem; font-weight: 500; font-family: 'DM Mono', monospace !important; color: #e3e3e3; margin: 0; line-height: 1; }
    .level-grade-label { font-size: 0.72rem; color: #8e918f; text-transform: uppercase; letter-spacing: 0.05em; }

    /* Text Stat Pills */
    .stat-row { display: flex; gap: 0.8rem; margin-top: 1rem; flex-wrap: wrap; }
    .stat-pill { background: #1e1f20; border: 1px solid #444746; border-radius: 10px; padding: 0.8rem 1.2rem; flex: 1; min-width: 100px; text-align: center; }
    .stat-pill-value { font-size: 1.4rem; font-weight: 500; color: #e3e3e3; font-family: 'DM Mono', monospace !important; display: block; }
    .stat-pill-label { font-size: 0.7rem; color: #8e918f; text-transform: uppercase; font-weight: 500; letter-spacing: 0.05em;}

    /* Tooltips */
    .tooltip-wrap { position: relative; display: inline-block; cursor: help; }
    .tooltip-icon { width: 14px; height: 14px; background: #282a2c; border-radius: 50%; display: inline-flex; align-items: center; justify-content: center; font-size: 0.6rem; color: #8e918f; }
    .tooltip-box { visibility: hidden; opacity: 0; background: #282a2c; border: 1px solid #444746; border-radius: 8px; padding: 0.8rem 1rem; width: 220px; position: absolute; bottom: 130%; left: 50%; transform: translateX(-50%); z-index: 9999; box-shadow: 0 4px 12px rgba(0,0,0,0.3); pointer-events: none; }
    .tooltip-wrap:hover .tooltip-box { visibility: visible; opacity: 1; }
    .tooltip-box p { color: #c4c7c5; font-size: 0.78rem; margin: 0; line-height: 1.5; text-transform: none; font-weight: 400; }
    .tooltip-box strong { color: #e3e3e3; display: block; margin-bottom: 0.3rem; font-size: 0.8rem; text-transform: none; }

    /* Tabs */
    .stTabs [data-baseweb="tab-list"] { background: transparent !important; border-bottom: 1px solid #444746 !important; border-radius: 0 !important; padding: 0 !important; }
    .stTabs [data-baseweb="tab"] { background: transparent !important; color: #8e918f !important; border-radius: 0 !important; padding: 0.8rem 1rem !important; border: none !important; }
    .stTabs [aria-selected="true"] { background: transparent !important; color: #a8c7fa !important; border-bottom: 2px solid #a8c7fa !important; }

    /* File Uploader */
    .stFileUploader > div { background: #1e1f20 !important; border: 1px dashed #444746 !important; border-radius: 8px !important; }

    /* Text Area */
    .stTextArea textarea { background: #1e1f20 !important; border: 1px solid #444746 !important; border-radius: 8px !important; color: #e3e3e3 !important; }
    .stTextArea textarea:focus { border-color: #a8c7fa !important; box-shadow: none !important; }
</style>
"""
print("styles.py created successfully!")

In [ ]:
%%writefile templates.py
class Templates:
    @staticmethod
    def logo():
        return """
        <div style='text-align:center; margin-bottom:1.5rem; padding-bottom:1rem;'>
            <img src='https://upload.wikimedia.org/wikipedia/commons/9/95/Infosys_logo.svg'
                 style='width:130px; margin-bottom:0.8rem;'>
            <p style='color:#8e918f; font-size:0.75rem; letter-spacing:0.05em;
                      text-transform:uppercase; margin:0;'>
                PolicyNav | Secure Access Management
            </p>
        </div>
        """
print("templates.py created!")

In [ ]:
%%writefile app.py
import streamlit as st
import sqlite3
import jwt
import datetime
import hashlib
import re
import time
import secrets
import smtplib
import base64
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import os
from styles import CSS
from templates import Templates
import readability_utils as readability
import PyPDF2
import streamlit.components.v1 as components

# --- DATA SCIENCE & ANALYTICS IMPORTS ---
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from wordcloud import WordCloud

# --- AI IMPORTS ---
import pickle
import faiss
import spacy
import torch
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pyvis.network import Network
import wikipedia
import knowledge_graph

# ================= LOAD SECRETS =================
EMAIL_ADDRESS = os.environ.get('EMAIL_ID', 'your_email@gmail.com')
EMAIL_PASSWORD = os.environ.get('EMAIL_APP_PASSWORD', 'your_app_password')
SECRET_KEY = os.environ.get('JWT_SECRET', 'fallback-secret-key-change-me')
ADMIN_EMAIL    = os.environ.get('ADMIN_EMAIL', 'admin@policynav.com')
ADMIN_PASSWORD = os.environ.get('ADMIN_PASSWORD', 'Admin@123')
# ================= PAGE CONFIG =================
st.set_page_config(page_title="PolicyNav", layout="wide", initial_sidebar_state="expanded")
st.markdown(CSS, unsafe_allow_html=True)

# Professional, Vibrant SaaS Sidebar CSS & Uploader Fix
st.markdown("""
<style>
[data-testid="stSidebar"] { background-color: #0b0f19; }
[data-testid="stSidebar"] .menu-label { color: #60a5fa; font-size: 0.85rem; text-transform: uppercase; letter-spacing: 2px; margin: 35px 0 15px 15px; font-weight: 700; border-bottom: 1px solid #1e293b; padding-bottom: 5px; }
[data-testid="stSidebar"] .stButton button { width: 90%; margin: 5px auto; border-radius: 10px; border: 1px solid transparent; text-align: left; justify-content: flex-start; padding: 14px 20px; font-weight: 500; color: #94a3b8; background-color: transparent; transition: all 0.4s cubic-bezier(0.4, 0, 0.2, 1); }
[data-testid="stSidebar"] .stButton button p { font-size: 1.1rem; }
[data-testid="stSidebar"] .stButton button:hover { background-color: #1e293b; color: #ffffff; transform: translateX(8px); border-left: 5px solid #3b82f6; box-shadow: 0 4px 15px rgba(0,0,0,0.3); }
[data-testid="stSidebar"] .stButton button[kind="primary"] { background: linear-gradient(90deg, #1e3a8a 0%, #1e40af 100%); color: #ffffff !important; font-weight: 600; border-left: 5px solid #60a5fa; box-shadow: 0 4px 20px rgba(37, 99, 235, 0.4); }

/* 👇 Absolute Override for Streamlit Uploader Text 👇 */
[data-testid="stFileUploadDropzone"] small,
[data-testid="stFileUploader"] small,
[data-testid="stFileUploadDropzone"] div[data-testid="stMarkdownContainer"] p,
[data-testid="stFileUploadDropzone"] > div > small,
section[data-testid="stFileUploadDropzone"] small {
    display: none !important;
    font-size: 0px !important;
    color: transparent !important;
}
</style>
""", unsafe_allow_html=True)

# ================= AI MODEL CACHING =================
@st.cache_resource
def load_data_and_models():
    index_path = "/content/drive/MyDrive/PolicyNav/policy_vector_db.index"
    chunks_path = "/content/drive/MyDrive/PolicyNav/chunks.pkl"
    if os.path.exists(index_path) and os.path.exists(chunks_path):
        index = faiss.read_index(index_path)
        with open(chunks_path, "rb") as f: chunks = pickle.load(f)
    else:
        index, chunks = None, [{"source": "No Data", "text": "Run PDF processor first."}]
    embed_model = SentenceTransformer("all-MiniLM-L6-v2")
    t_tokenizer = AutoTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
    t_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/nllb-200-distilled-600M", device_map="auto")
    s_tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
    s_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base", device_map="auto")
    nlp = spacy.load("en_core_web_sm")
    return index, chunks, embed_model, t_tokenizer, t_model, s_tokenizer, s_model, nlp

# ================= AI HELPER FUNCTIONS =================
LANG_CODES = {"English": "eng_Latn", "Hindi": "hin_Deva", "Tamil": "tam_Taml", "Telugu": "tel_Telu", "Kannada": "kan_Knda", "Marathi": "mar_Deva", "Bengali": "ben_Beng", "Malayalam": "mal_Mlym", "Gujarati": "guj_Gujr", "Urdu": "urd_Arab"}

def translate_fast(text, source_lang, target_lang, t_tokenizer, t_model):
    if source_lang == target_lang: return text
    t_tokenizer.src_lang = LANG_CODES.get(source_lang, "eng_Latn")
    inputs = t_tokenizer(text, return_tensors="pt", truncation=True).to(t_model.device)
    tgt_id = t_tokenizer.convert_tokens_to_ids(LANG_CODES.get(target_lang, "eng_Latn"))
    outputs = t_model.generate(**inputs, forced_bos_token_id=tgt_id, max_length=512)
    return t_tokenizer.decode(outputs[0], skip_special_tokens=True)

def search_policy(query, index, chunks, embed_model, top_k=5):
    if not index: return ["No database found."]
    query_vector = embed_model.encode([query])
    distances, indices = index.search(query_vector, top_k)
    return [chunks[i]["text"] for i in indices[0]]

def generate_knowledge_graph(chunks_subset, nlp):
    import knowledge_graph as _kg
    return _kg.generate_knowledge_graph(chunks_subset, nlp)

def global_web_research(query):
    results = []
    try:
        search_results = wikipedia.search(query, results=3)
        for title in search_results:
            summary = wikipedia.summary(title, sentences=2, auto_suggest=False)
            url = wikipedia.page(title, auto_suggest=False).url
            results.append({"title": title, "body": summary, "href": url})
    except:
        results.append({"title": "Notice", "body": "Could not fetch results.", "href": ""})
    return results

# ================= AUTH & DB CONFIG =================
ALGORITHM, TOKEN_EXPIRE_MINUTES, MAX_LOGIN_ATTEMPTS, LOCKOUT_TIME, OTP_EXPIRY_MINUTES = "HS256", 30, 3, 300, 10
DB_PATH = "/content/drive/MyDrive/PolicyNav/users.db"
conn   = sqlite3.connect(DB_PATH, check_same_thread=False)
cursor = conn.cursor()

cursor.execute("""CREATE TABLE IF NOT EXISTS users (id INTEGER PRIMARY KEY AUTOINCREMENT, username TEXT NOT NULL, email TEXT UNIQUE NOT NULL, password TEXT NOT NULL, security_question TEXT NOT NULL, security_answer TEXT NOT NULL, created_at TEXT, is_blocked INTEGER DEFAULT 0)""")
try: cursor.execute("ALTER TABLE users ADD COLUMN is_blocked INTEGER DEFAULT 0"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE users ADD COLUMN avatar TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE users ADD COLUMN role TEXT DEFAULT 'user'"); conn.commit()
except: pass

cursor.execute("""CREATE TABLE IF NOT EXISTS password_history (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT NOT NULL, password TEXT NOT NULL, set_at TEXT, FOREIGN KEY(email) REFERENCES users(email))""")
cursor.execute("""CREATE TABLE IF NOT EXISTS login_attempts (email TEXT PRIMARY KEY, attempts INTEGER DEFAULT 0, last_attempt REAL)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS otp_requests (email TEXT PRIMARY KEY, otp TEXT, expires_at REAL)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS user_activity (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, action_type TEXT, details TEXT, timestamp TEXT)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, rating INTEGER, comments TEXT, timestamp TEXT)""")
cursor.execute("""CREATE TABLE IF NOT EXISTS tool_feedback (id INTEGER PRIMARY KEY AUTOINCREMENT, email TEXT, tool_name TEXT, prompt TEXT, response TEXT, rating TEXT, comments TEXT, timestamp TEXT)""")

try: cursor.execute("ALTER TABLE tool_feedback ADD COLUMN comments TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE user_activity ADD COLUMN prompt TEXT"); conn.commit()
except: pass
try: cursor.execute("ALTER TABLE user_activity ADD COLUMN response TEXT"); conn.commit()
except: pass
conn.commit()

# ================= UTILS & VALIDS =================
def _get_timestamp(): return datetime.datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
def hash_password(password): return hashlib.sha256(password.encode()).hexdigest()
def create_token(email, username, role="user"): return jwt.encode({"sub": email, "username": username, "role": role, "exp": datetime.datetime.utcnow() + datetime.timedelta(minutes=TOKEN_EXPIRE_MINUTES)}, SECRET_KEY, algorithm=ALGORITHM)
def valid_email(email): return re.match(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$', email)

# ================= DB LOGIC =================
def log_activity(email, action_type, details, prompt="", response=""):
    cursor.execute("INSERT INTO user_activity (email, action_type, details, prompt, response, timestamp) VALUES (?, ?, ?, ?, ?, ?)", (email, action_type, details, prompt, response, _get_timestamp())); conn.commit()

def get_user_history(email):
    cursor.execute("SELECT action_type, details, prompt, response, timestamp FROM user_activity WHERE email = ? AND action_type NOT IN ('System', 'Security', 'Profile') ORDER BY id DESC", (email,))
    return cursor.fetchall()

def submit_feedback(email, rating, comments):
    cursor.execute("INSERT INTO feedback (email, rating, comments, timestamp) VALUES (?, ?, ?, ?)", (email, rating, comments, _get_timestamp())); conn.commit()

def submit_tool_feedback(email, tool_name, prompt, response, rating, comments):
    cursor.execute("INSERT INTO tool_feedback (email, tool_name, prompt, response, rating, comments, timestamp) VALUES (?, ?, ?, ?, ?, ?, ?)", (email, tool_name, prompt, response, rating, comments, _get_timestamp()))
    conn.commit()

def get_all_tool_feedback():
    cursor.execute("SELECT email, tool_name, prompt, response, rating, comments, timestamp FROM tool_feedback ORDER BY id DESC")
    return cursor.fetchall()

def get_user_avatar(email):
    cursor.execute("SELECT avatar FROM users WHERE email = ?", (email,))
    res = cursor.fetchone()
    return res[0] if res else None

def update_avatar(email, base64_string):
    cursor.execute("UPDATE users SET avatar = ? WHERE email = ?", (base64_string, email))
    conn.commit()

def update_user_email_db(old_email, new_email):
    cursor.execute("UPDATE users SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE password_history SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE login_attempts SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE otp_requests SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE user_activity SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE feedback SET email = ? WHERE email = ?", (new_email, old_email))
    cursor.execute("UPDATE tool_feedback SET email = ? WHERE email = ?", (new_email, old_email))
    conn.commit()

def get_all_feedback(): cursor.execute("SELECT email, rating, comments, timestamp FROM feedback ORDER BY id DESC"); return cursor.fetchall()
def get_login_attempts(email): cursor.execute("SELECT attempts, last_attempt FROM login_attempts WHERE email = ?", (email,)); data = cursor.fetchone(); return data if data else (0, 0)
def increment_login_attempts(email): attempts, _ = get_login_attempts(email); cursor.execute("INSERT OR REPLACE INTO login_attempts (email, attempts, last_attempt) VALUES (?, ?, ?)", (email, attempts + 1, time.time())); conn.commit()
def reset_login_attempts(email): cursor.execute("DELETE FROM login_attempts WHERE email = ?", (email,)); conn.commit()

def is_rate_limited(email):
    attempts, last_attempt = get_login_attempts(email)
    if attempts >= MAX_LOGIN_ATTEMPTS:
        if time.time() - last_attempt < LOCKOUT_TIME: return True, LOCKOUT_TIME - (time.time() - last_attempt)
        else: reset_login_attempts(email)
    return False, 0

def register_user(username, email, password, security_question, security_answer):
    try:
        now, hashed_pass, hashed_answer = _get_timestamp(), hash_password(password), hash_password(security_answer.strip())
        cursor.execute("INSERT INTO users (username, email, password, security_question, security_answer, created_at) VALUES (?, ?, ?, ?, ?, ?)", (username, email, hashed_pass, security_question, hashed_answer, now))
        cursor.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed_pass, now)); conn.commit(); return True
    except sqlite3.IntegrityError: return False

# 👇 FIXED 1: Admins are fully exempt from login lockouts 👇
def authenticate_user(email, password):
    cursor.execute("SELECT username, password, is_blocked, role FROM users WHERE email = ?", (email,))
    user = cursor.fetchone()

    # Check if they are the hardcoded admin or promoted admin
    is_admin = (email == ADMIN_EMAIL) or (user and user[3] == 'admin')

    # Bypass rate limits for Admins completely
    if not is_admin:
        is_limited, _ = is_rate_limited(email)
        if is_limited: return False, "locked"

    if user:
        if user[2] == 1: return False, "blocked"
        if user[1] == hash_password(password):
            reset_login_attempts(email)
            return True, (user[0], user[3])

    # Only increment failed attempts for standard users
    if not is_admin:
        increment_login_attempts(email)
    return False, None

def authenticate_admin(email, password): return email == ADMIN_EMAIL and password == ADMIN_PASSWORD
def check_user_exists(email): cursor.execute("SELECT 1 FROM users WHERE email = ?", (email,)); return cursor.fetchone() is not None
def get_user_details(email): cursor.execute("SELECT username, security_question, security_answer FROM users WHERE email = ?", (email,)); return cursor.fetchone()
def check_password_reused(email, new_password):
    cursor.execute("SELECT password FROM password_history WHERE email = ? ORDER BY id DESC LIMIT 5", (email,)); history = cursor.fetchall(); hashed_new = hash_password(new_password)
    for (stored_hash,) in history:
        if stored_hash == hashed_new: return True
    return False
def update_password(email, new_password):
    hashed, now = hash_password(new_password), _get_timestamp()
    cursor.execute("UPDATE users SET password = ? WHERE email = ?", (hashed, email)); cursor.execute("INSERT INTO password_history (email, password, set_at) VALUES (?, ?, ?)", (email, hashed, now)); conn.commit(); reset_login_attempts(email)
def verify_security_answer(email, answer): cursor.execute("SELECT security_answer FROM users WHERE email = ?", (email,)); stored_answer = cursor.fetchone(); return stored_answer and stored_answer[0] == hash_password(answer.strip())

def get_all_users(): cursor.execute("SELECT id, username, email, created_at, is_blocked, role FROM users ORDER BY id DESC"); return cursor.fetchall()
def toggle_block_user(email, current_status):
    new_status = 0 if current_status == 1 else 1
    cursor.execute("UPDATE users SET is_blocked = ? WHERE email = ?", (new_status, email)); conn.commit()
def toggle_role_user(email, current_role):
    new_role = 'admin' if current_role == 'user' else 'user'
    cursor.execute("UPDATE users SET role = ? WHERE email = ?", (new_role, email)); conn.commit()

# 👇 FIXED 2: Soft-delete logic. Anonymizes Data instead of wiping it 👇
def delete_user(email):
    # 1. Anonymize/Retain their valuable data for Admin Analytics
    deleted_label = f"Deleted User ({email})"
    cursor.execute("UPDATE user_activity SET email = ? WHERE email = ?", (deleted_label, email))
    cursor.execute("UPDATE feedback SET email = ? WHERE email = ?", (deleted_label, email))
    cursor.execute("UPDATE tool_feedback SET email = ? WHERE email = ?", (deleted_label, email))

    # 2. Completely remove their actual account and login access
    cursor.execute("DELETE FROM users WHERE email = ?", (email,))
    cursor.execute("DELETE FROM password_history WHERE email = ?", (email,))
    cursor.execute("DELETE FROM login_attempts WHERE email = ?", (email,))
    cursor.execute("DELETE FROM otp_requests WHERE email = ?", (email,))
    conn.commit()

def get_user_stats():
    cursor.execute("SELECT COUNT(*) FROM users"); total = cursor.fetchone()[0]; cursor.execute("SELECT COUNT(*) FROM users WHERE is_blocked = 1"); blocked = cursor.fetchone()[0]
    return total, blocked, total - blocked

def generate_otp(): return f"{secrets.randbelow(1000000):06d}"
def save_otp(email, otp): expires_at = time.time() + (OTP_EXPIRY_MINUTES * 60); cursor.execute("INSERT OR REPLACE INTO otp_requests (email, otp, expires_at) VALUES (?, ?, ?)", (email, otp, expires_at)); conn.commit()
def verify_otp(email, otp):
    cursor.execute("SELECT otp, expires_at FROM otp_requests WHERE email = ?", (email,)); data = cursor.fetchone()
    if data and data[0] == otp and time.time() < data[1]: cursor.execute("DELETE FROM otp_requests WHERE email = ?", (email,)); conn.commit(); return True
    return False
def send_otp_email(to_email, otp):
    try:
        msg = MIMEMultipart(); msg['From'] = f"PolicyNav <{EMAIL_ADDRESS}>"; msg['To'] = to_email; msg['Subject'] = "PolicyNav Password Reset"
        body = f"<html><body><h1 style='color: #e3e3e3; text-align: center;'>{otp}</h1></body></html>"
        msg.attach(MIMEText(body, 'html')); server = smtplib.SMTP('smtp.gmail.com', 587); server.starttls(); server.login(EMAIL_ADDRESS, EMAIL_PASSWORD); server.send_message(msg); server.quit()
        return True, "OTP sent successfully"
    except Exception as e: return False, str(e)

# ================= SESSION =================
# Added variables to track the Email Update OTP flow
_session_defaults = {"page": "login", "token": None, "role": "user", "reset_email": None, "security_question": None, "otp_verified": False, "username": None, "email": None, "menu_option": "Dashboard", "reset_method": None, "otp_sent": False, "email_update_otp_sent": False, "pending_new_email": None}
for _k, _v in _session_defaults.items():
    if _k not in st.session_state: st.session_state[_k] = _v

def metric_card(label, tooltip_desc, value, bar_pct, bar_color):
    bar_pct_clamped = max(0, min(100, bar_pct))
    return f"""<div class="metric-card"><div class="metric-title">{label}</div><div class="metric-value">{value:.1f}</div><div class="metric-bar-track"><div class="metric-bar-fill" style="width:{bar_pct_clamped}%; background:{bar_color};"></div></div><p class="metric-interpretation">{tooltip_desc}</p></div>"""

# ================= PAGES =================
def dashboard_page(username, chunks):
    now_dt = datetime.datetime.now()
    _, center_col, _ = st.columns([1, 6, 1])
    with center_col:
        st.markdown(f"<div style='margin-bottom: 2rem; margin-top: 2rem; text-align: center;'><p style='color: #a8c7fa; font-size: 0.85rem; margin: 0 0 0.2rem 0; text-transform: uppercase; letter-spacing: 0.05em;'>Welcome back</p><h1 style='margin-bottom: 0.2rem; font-size: 2.5rem; color: #e3e3e3;'>{username}</h1><p style='color: #8e918f; font-size: 0.9rem; margin: 0;'>{now_dt.strftime('%A, %B %d, %Y')}</p></div>", unsafe_allow_html=True)

        st.markdown("<h3 style='text-align: center; color: #e3e3e3; margin-top: 1rem;'>Latest Policy Updates</h3>", unsafe_allow_html=True)
        st.markdown("<p style='color: #8e918f; font-size: 0.9rem; text-align: center; margin-bottom: 1.5rem;'>Hover over the feed to pause scrolling and read.</p>", unsafe_allow_html=True)

        unique_srcs = []
        for c in reversed(chunks):
            if c["source"] not in unique_srcs and c["source"] != "No Data":
                unique_srcs.append(c["source"])
            if len(unique_srcs) >= 5: break

        cards_html = ""
        if not unique_srcs:
            cards_html = "<div class='policy-card'><p class='policy-title'>No Policies Found</p><p class='policy-desc'>Please process PDFs to populate the database.</p></div>"
        else:
            for src in unique_srcs:
                first_chunk = ""
                for c in chunks:
                    if c["source"] == src:
                        first_chunk = c["text"]
                        break
                title = src.replace(".pdf", "").replace("_", " ").title()
                desc = first_chunk[:130] + "..." if len(first_chunk) > 130 else first_chunk
                cards_html += f"<div class='policy-card'><p class='policy-title'>{title}</p><p class='policy-desc'>{desc}</p><span class='policy-tag'>Recently Added to Database</span></div>"

        ticker_css = "<style>.news-ticker-container {height: 350px; overflow: hidden; background-color: #1e1f20; border-radius: 12px; border: 1px solid #444746; position: relative; padding: 10px 20px; box-shadow: 0 10px 20px rgba(0,0,0,0.4); } .news-scroll {display: flex; flex-direction: column; gap: 15px; animation: scroll-vertical 20s linear infinite; } .news-ticker-container:hover .news-scroll {animation-play-state: paused; } .policy-card {background-color: #131314; padding: 18px; border-radius: 8px; border-left: 5px solid #a8c7fa; box-shadow: 0 4px 6px rgba(0,0,0,0.1); margin-bottom:10px;} .policy-title {color: #a8c7fa; font-size: 1.15rem; font-weight: 600; margin: 0 0 8px 0; } .policy-desc {color: #e3e3e3; font-size: 0.95rem; margin: 0; line-height: 1.5; } .policy-tag {display: inline-block; background-color: #2e3032; color: #81c995; font-size: 0.75rem; padding: 3px 8px; border-radius: 4px; margin-top: 10px; } @keyframes scroll-vertical {0% { transform: translateY(100%); } 100% { transform: translateY(-100%); } } </style>"
        ticker_html = f"<div class='news-ticker-container'><div class='news-scroll'>{cards_html}</div></div>"

        st.markdown(ticker_css + ticker_html, unsafe_allow_html=True)

def signup():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Create Account</h1>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        col1, col2 = st.columns(2)
        with col1:
            username = st.text_input("Username", placeholder="your_username", key="signup_username")
            password = st.text_input("Password", type="password", placeholder="••••••••", key="signup_password")

            # 👇 NEW: Live Password Strength for Signup 👇
            is_len = len(password) >= 6
            is_upper = bool(re.search(r"[A-Z]", password))
            is_num = bool(re.search(r"\d", password))
            is_spec = bool(re.search(r"[^A-Za-z0-9]", password))

            if password:
                # Using <br> here so it fits nicely inside the half-column!
                st.markdown(f"<p style='font-size:0.75rem; margin-top:-10px; margin-bottom:10px; color:#c4c7c5; line-height:1.4;'>" +
                            (f"<span style='color:#81c995;'>✅ 6+ chars</span>" if is_len else "❌ 6+ chars") + " &nbsp;|&nbsp; " +
                            (f"<span style='color:#81c995;'>✅ Uppercase</span>" if is_upper else "❌ Uppercase") + "<br>" +
                            (f"<span style='color:#81c995;'>✅ Number</span>" if is_num else "❌ Number") + " &nbsp;|&nbsp; " +
                            (f"<span style='color:#81c995;'>✅ Special</span>" if is_spec else "❌ Special") +
                            "</p>", unsafe_allow_html=True)

            security_question = st.selectbox("Security Question", ["What is your pet name?", "What is your mother's maiden name?", "What is your favorite teacher?"], key="signup_question")
        with col2:
            email = st.text_input("Email", placeholder="you@example.com", key="signup_email")
            confirm = st.text_input("Confirm Password", type="password", placeholder="••••••••", key="signup_confirm")
            security_answer = st.text_input("Security Answer", placeholder="Your answer", key="signup_answer")

        if st.button("Create Account", key="signup_button", use_container_width=True, type="primary"):
            if not all([username, email, password, confirm, security_answer]): st.error("All fields required"); return
            if password != confirm: st.error("Passwords do not match"); return

            # 👇 NEW: Backend block to prevent creating weak accounts 👇
            if not (is_len and is_upper and is_num and is_spec):
                st.error("Password is too weak. Please meet all the security requirements.")
                return

            if register_user(username, email, password, security_question, security_answer):
                st.success("Account created successfully"); time.sleep(1.5); st.session_state.page = "login"; st.rerun()
            else: st.error("Registration failed or Email exists")
        if st.button("Back to Login", key="back_to_login", use_container_width=True): st.session_state.page = "login"; st.rerun()

def login():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Welcome back</h1>", unsafe_allow_html=True)
    st.markdown("<p style='text-align: center; margin-bottom: 2rem;'>Sign in to your account</p>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        email = st.text_input("Email", placeholder="you@example.com", key="login_email")
        password = st.text_input("Password", type="password", placeholder="••••••••", key="login_password")
        if st.button("Sign In", key="login_button", use_container_width=True, type="primary"):
            if not email or not password:
                st.error("Please enter email and pass both")
            elif authenticate_admin(email, password):
                st.session_state.token = "admin_token"
                st.session_state.username = "Admin"
                st.session_state.email = email
                st.session_state.role = "admin"
                st.session_state.page = "dashboard"
                st.session_state.menu_option = "Dashboard"
                st.rerun()
            else:
                auth_result, status = authenticate_user(email, password)
                if auth_result:
                    username, role = status
                    st.session_state.token = create_token(email, username, role)
                    st.session_state.username = username
                    st.session_state.email = email
                    st.session_state.role = role
                    st.session_state.page = "dashboard"
                    st.session_state.menu_option = "Dashboard"
                    log_activity(email, "System", "User logged in successfully")
                    st.rerun()
                else:
                    if status == "locked": st.error("Account locked. Try again later.")
                    elif status == "blocked": st.error("Account blocked by admin.")
                    else: st.error("Invalid credentials")
        st.markdown("<br>", unsafe_allow_html=True)
        c1, c2 = st.columns(2)
        with c1:
            if st.button("Create Account", use_container_width=True): st.session_state.page = "signup"; st.rerun()
        with c2:
            if st.button("Forgot Password", use_container_width=True): st.session_state.page = "forgot"; st.rerun()

def forgot_password():
    st.markdown(Templates.logo(), unsafe_allow_html=True)
    st.markdown("<h1 style='text-align: center;'>Reset Password</h1>", unsafe_allow_html=True)
    _, center_col, _ = st.columns([1, 2, 1])
    with center_col:
        if not st.session_state.reset_email:
            email = st.text_input("Email", placeholder="Your registered email", key="forgot_email")
            st.markdown("<p style='text-align:center; margin:1rem 0 0.5rem; font-size:0.85rem;'>Choose your verification method</p>", unsafe_allow_html=True)
            col_otp, col_sec = st.columns(2)
            with col_otp: otp_btn = st.button("Via OTP Email", use_container_width=True)
            with col_sec: sec_btn = st.button("Via Security Q&A", use_container_width=True)
            if otp_btn or sec_btn:
                if not email: st.error("Please enter your email first")
                elif not check_user_exists(email): st.error("Email not found in our system")
                elif otp_btn:
                    otp = generate_otp(); save_otp(email, otp); success, msg = send_otp_email(email, otp)
                    if success: st.session_state.reset_email = email; st.session_state.reset_method = "otp"; st.success("OTP sent to your email."); time.sleep(1); st.rerun()
                    else: st.error(f"Failed to send OTP: {msg}")
                else: st.session_state.reset_email = email; st.session_state.reset_method = "security"; st.rerun()
        elif st.session_state.reset_method == "otp" and not st.session_state.otp_verified:
            otp_input = st.text_input("Enter OTP", placeholder="6-digit code", max_chars=6)
            if st.button("Verify OTP", type="primary", use_container_width=True):
                if verify_otp(st.session_state.reset_email, otp_input): st.session_state.otp_verified = True; st.success("OTP verified."); st.rerun()
                else: st.error("Invalid or expired OTP")
        elif st.session_state.reset_method == "security" and not st.session_state.otp_verified:
            user_details = get_user_details(st.session_state.reset_email)
            if user_details:
                st.info(f"Question: {user_details[1]}")
                answer = st.text_input("Your Answer", placeholder="Enter your answer")
                if st.button("Verify Answer", type="primary", use_container_width=True):
                    if verify_security_answer(st.session_state.reset_email, answer): st.session_state.otp_verified = True; st.success("Answer verified."); time.sleep(0.8); st.rerun()
                    else: st.error("Incorrect security answer")
        elif st.session_state.otp_verified:
            new_password = st.text_input("New Password", type="password", placeholder="••••••••")

            # 👇 NEW: Live Password Strength for Forgot Password 👇
            is_len = len(new_password) >= 6
            is_upper = bool(re.search(r"[A-Z]", new_password))
            is_num = bool(re.search(r"\d", new_password))
            is_spec = bool(re.search(r"[^A-Za-z0-9]", new_password))

            if new_password:
                st.markdown(f"<p style='font-size:0.75rem; margin-top:-10px; margin-bottom:10px; color:#c4c7c5;'>" +
                            (f"<span style='color:#81c995;'>✅ 6+ chars</span>" if is_len else "❌ 6+ chars") + " &nbsp;|&nbsp; " +
                            (f"<span style='color:#81c995;'>✅ Uppercase</span>" if is_upper else "❌ Uppercase") + " &nbsp;|&nbsp; " +
                            (f"<span style='color:#81c995;'>✅ Number</span>" if is_num else "❌ Number") + " &nbsp;|&nbsp; " +
                            (f"<span style='color:#81c995;'>✅ Special</span>" if is_spec else "❌ Special") +
                            "</p>", unsafe_allow_html=True)

            confirm_password = st.text_input("Confirm Password", type="password", placeholder="••••••••")

            if st.button("Reset Password", type="primary", use_container_width=True):
                # 👇 NEW: Backend block to prevent weak resets 👇
                if not (is_len and is_upper and is_num and is_spec):
                    st.error("Password is too weak. Please meet all security requirements.")
                elif new_password == confirm_password and not check_password_reused(st.session_state.reset_email, new_password):
                    update_password(st.session_state.reset_email, new_password); st.success("Password updated successfully."); time.sleep(1.5)
                    st.session_state.page = "login"; st.session_state.reset_email = None; st.session_state.otp_verified = False; st.rerun()
                else: st.error("Passwords mismatch or reused old password")
        st.markdown("<br>", unsafe_allow_html=True)

# ================= MAIN ROUTING =================
if st.session_state.token:
    is_admin = st.session_state.role == "admin"
    username = st.session_state.username
    if not is_admin:
        with st.spinner("Initializing Workspace..."): index, chunks, embed_model, t_tokenizer, t_model, s_tokenizer, s_model, nlp = load_data_and_models()

    with st.sidebar:
        st.markdown("<div style='display:flex; align-items:center; gap:15px; margin-bottom:2rem; padding: 0 10px;'><img src='https://upload.wikimedia.org/wikipedia/commons/9/95/Infosys_logo.svg' style='width:75px;'/><span style='color:#ffffff;font-weight:700;font-size:1.3rem;letter-spacing:-0.02em;'>PolicyNav</span></div>", unsafe_allow_html=True)
        if is_admin:
            st.markdown("<div class='menu-label'>ADMINISTRATION</div>", unsafe_allow_html=True)
            if st.button("System Dashboard", use_container_width=True, type="primary" if st.session_state.menu_option == "Dashboard" else "secondary"): st.session_state.menu_option = "Dashboard"; st.rerun()
        else:
            st.markdown("<div class='menu-label'>AI INTELLIGENCE</div>", unsafe_allow_html=True)
            if st.button("User Dashboard", use_container_width=True, type="primary" if st.session_state.menu_option == "Dashboard" else "secondary"): st.session_state.menu_option = "Dashboard"; st.rerun()
            if st.button("AI Policy Assistant", use_container_width=True, type="primary" if st.session_state.menu_option == "Chat" else "secondary"): st.session_state.menu_option = "Chat"; st.rerun()
            if st.button("AI Policy Summarizer", use_container_width=True, type="primary" if st.session_state.menu_option == "Summarization" else "secondary"): st.session_state.menu_option = "Summarization"; st.rerun()
            if st.button("Entity Knowledge Graph", use_container_width=True, type="primary" if st.session_state.menu_option == "Graph" else "secondary"): st.session_state.menu_option = "Graph"; st.rerun()
            if st.button("Global Web Search", use_container_width=True, type="primary" if st.session_state.menu_option == "Web" else "secondary"): st.session_state.menu_option = "Web"; st.rerun()
            if st.button("Readability Analyzer", use_container_width=True, type="primary" if st.session_state.menu_option == "Readability" else "secondary"): st.session_state.menu_option = "Readability"; st.rerun()
            if st.button("Language Translator", use_container_width=True, type="primary" if st.session_state.menu_option == "Translation" else "secondary"): st.session_state.menu_option = "Translation"; st.rerun()

            st.markdown("<div class='menu-label'>PERSONAL PORTAL</div>", unsafe_allow_html=True)
            if st.button("My Activity History", use_container_width=True, type="primary" if st.session_state.menu_option == "History" else "secondary"): st.session_state.menu_option = "History"; st.rerun()
            if st.button("Submit Feedback", use_container_width=True, type="primary" if st.session_state.menu_option == "Feedback" else "secondary"): st.session_state.menu_option = "Feedback"; st.rerun()
            if st.button("User Profile", use_container_width=True, type="primary" if st.session_state.menu_option == "Profile" else "secondary"): st.session_state.menu_option = "Profile"; st.rerun()

        st.markdown("<div style='flex-grow: 1;'></div>", unsafe_allow_html=True)
        if st.button("Log out", key="logout_btn", use_container_width=True):
            log_activity(st.session_state.email, "System", "User logged out")
            for key in list(st.session_state.keys()): del st.session_state[key]
            st.rerun()

        user_avatar = None
        if not is_admin: user_avatar = get_user_avatar(st.session_state.email)

        if user_avatar: avatar_html = f"<img src='data:image/png;base64,{user_avatar}' style='width:28px; height:28px; border-radius:50%; margin-right:10px; object-fit:cover; border: 1px solid #444746;'/>"
        else: avatar_html = f"<div style='width:24px; height:24px; border-radius:50%; background:#444746; display:flex; align-items:center; justify-content:center; margin-right:10px; font-size:0.7rem; color:#e3e3e3;'>{username[0].upper()}</div>"
        st.markdown(f"<div style='padding:20px; border-top:1px solid #1e293b; color:#94a3b8; font-size:0.9rem;'>Logged in as: <br><div style='display:flex; align-items:center; margin-top:8px;'>{avatar_html}<b style='color:#ffffff;'>{username}</b></div></div>", unsafe_allow_html=True)

    # --- MAIN CONTENT AREA ROUTING ---
    if is_admin:
        if st.session_state.menu_option == "Dashboard":
            st.markdown("<h2>Admin Command Center</h2>", unsafe_allow_html=True)

            # 👇 FIXED: Added a 5th tab for "Activity Logs" 👇
            tab1, tab2, tab3, tab4, tab5 = st.tabs(["User Management", "System Analytics", "Feedback Analysis", "Activity Logs", "Data Exports"])

            with tab1:
                total, blocked, active = get_user_stats()
                c1, c2, c3 = st.columns(3)
                with c1: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#e3e3e3;'>{total}</p><p class='card-meta'>Total Users</p></div>", unsafe_allow_html=True)
                with c2: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#a8c7fa;'>{active}</p><p class='card-meta'>Active Users</p></div>", unsafe_allow_html=True)
                with c3: st.markdown(f"<div class='clean-card'><p style='font-size:2rem; margin:0; color:#e3e3e3;'>{blocked}</p><p class='card-meta'>Blocked Accounts</p></div>", unsafe_allow_html=True)

                users = get_all_users()
                for uid, uname, uemail, ucreated, ublocked, urole in users:
                    is_locked, _ = is_rate_limited(uemail)
                    needs_unlock = ublocked == 1 or is_locked

                    st.markdown(f"<div class='clean-card' style='margin-bottom:10px; padding:15px;'>", unsafe_allow_html=True)
                    colA, colB, colC, colD = st.columns([3, 2, 2, 2])
                    with colA:
                        locked_text = "<span style='color:#f28b82;'>YES (Temp)</span>" if is_locked else "NO"
                        blocked_text = "<span style='color:#f28b82;'>YES (Manual)</span>" if ublocked else "NO"
                        st.markdown(f"**{uname}** ({uemail})<br><span style='color:#8e918f; font-size:0.8rem;'>Role: {urole.upper()} | Blocked: {blocked_text} | Locked: {locked_text}</span>", unsafe_allow_html=True)
                    with colB:
                        if st.button("Promote to Admin" if urole == 'user' else "Demote to User", key=f"role_{uid}", use_container_width=True):
                            toggle_role_user(uemail, urole); st.rerun()
                    with colC:
                        if st.button("Unlock Account" if needs_unlock else "Block Account", key=f"lock_{uid}", use_container_width=True):
                            if needs_unlock:
                                toggle_block_user(uemail, 0)
                                reset_login_attempts(uemail)
                            else:
                                toggle_block_user(uemail, 1)
                            st.rerun()
                    with colD:
                        if st.button("Delete User", key=f"del_{uid}", type="primary", use_container_width=True):
                            delete_user(uemail); st.rerun()
                    st.markdown("</div>", unsafe_allow_html=True)

            with tab2:
                st.markdown("### Real-Time Platform Analytics")
                try:
                    df = pd.read_sql("SELECT action_type, details FROM user_activity WHERE action_type NOT IN ('System', 'Security', 'Profile')", conn)
                    if not df.empty:
                        c1, c2 = st.columns(2)
                        with c1:
                            feat_counts = df['action_type'].value_counts().reset_index()
                            feat_counts.columns = ['AI Feature', 'Total Uses']
                            fig1 = px.bar(feat_counts, x='AI Feature', y='Total Uses', title="Popularity of AI Tools", template='plotly_dark', color_discrete_sequence=['#60a5fa'])
                            fig1.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
                            st.plotly_chart(fig1, use_container_width=True)
                        with c2:
                            known_langs = list(LANG_CODES.keys())
                            df['Language'] = df['details'].apply(lambda x: next((l for l in known_langs if l.lower() in str(x).lower()), 'English'))
                            lang_counts = df['Language'].value_counts().reset_index()
                            lang_counts.columns = ['Language', 'Total Output']
                            fig2 = px.pie(lang_counts, names='Language', values='Total Output', title="Regional Language Utilization", template='plotly_dark')
                            fig2.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
                            st.plotly_chart(fig2, use_container_width=True)
                    else:
                        st.info("Not enough user activity data to generate graphs yet.")
                except Exception as e:
                    st.error(f"Could not load analytics: {e}")

            with tab3:
                st.markdown("### User Feedback WordCloud")
                try:
                    fb_df = pd.read_sql("SELECT comments FROM feedback WHERE comments IS NOT NULL AND comments != ''", conn)
                    t_fb_df = pd.read_sql("SELECT comments FROM tool_feedback WHERE comments IS NOT NULL AND comments != ''", conn)
                    all_text = " ".join(fb_df['comments'].tolist() + t_fb_df['comments'].tolist())

                    if all_text.strip():
                        wordcloud = WordCloud(width=800, height=350, background_color='#131314', colormap='Blues', max_words=100).generate(all_text)
                        fig, ax = plt.subplots(figsize=(10, 4.5), facecolor='#131314')
                        ax.imshow(wordcloud, interpolation='bilinear')
                        ax.axis('off')
                        st.pyplot(fig)
                    else:
                        st.info("No written comments submitted by users yet to generate WordCloud.")
                except Exception as e:
                    st.error(f"Error generating WordCloud: {e}")

                st.markdown("---")
                c1, c2 = st.columns(2)
                with c1:
                    st.markdown("#### General Platform Feedback")
                    feedbacks = get_all_feedback()
                    if not feedbacks: st.info("No general feedback submitted yet.")
                    for email, rating, comments, ts in feedbacks:
                        st.markdown(f"<div class='clean-card'><p class='card-title'>{email} <span style='float:right; color:#8e918f; font-size:0.8rem;'>{ts}</span></p><p style='color:#a8c7fa; margin:5px 0;'>Rating: {rating}/5</p><p class='card-text'>{comments}</p></div>", unsafe_allow_html=True)
                with c2:
                    st.markdown("#### Individual AI Tool Tuning Data")
                    tool_feeds = get_all_tool_feedback()
                    if not tool_feeds: st.info("No specific AI tool feedback submitted yet.")
                    for email, tool_name, prompt, resp, rating, comments, ts in tool_feeds:
                        st.markdown(f"<div class='clean-card' style='margin-bottom:15px;'><div style='display:flex; justify-content:space-between;'><b style='color:#60a5fa; font-size:0.9rem; text-transform:uppercase;'>{tool_name}</b><span style='color:#8e918f; font-size:0.75rem;'>{ts}</span></div><p style='margin:5px 0; color:#e3e3e3; font-size:0.85rem;'><b>Rating:</b> <span style='color:#81c995;'>{rating}</span></p><p style='font-size:0.85rem; color:#e3e3e3; margin:0;'><b>Notes:</b> {comments if comments else '<i style="color:#8e918f;">No written comment</i>'}</p></div>", unsafe_allow_html=True)

            # 👇 NEW: Tab 4 - Global Activity Logs (Search & Filter) 👇
            with tab4:
                st.markdown("### Global User Activity Logs")
                st.markdown("<p style='color:#8e918f; font-size:0.9rem;'>Search or select a user to view their complete interaction history.</p>", unsafe_allow_html=True)

                try:
                    df_activity = pd.read_sql("SELECT * FROM user_activity ORDER BY id DESC", conn)
                    if not df_activity.empty:
                        # Creates the scrollable & searchable dropdown box!
                        user_list = ["All Users"] + df_activity['email'].unique().tolist()
                        selected_user = st.selectbox("Search or Select User Email:", user_list)

                        # Filter the database based on the admin's selection
                        if selected_user != "All Users":
                            filtered_df = df_activity[df_activity['email'] == selected_user]
                        else:
                            filtered_df = df_activity

                        st.markdown(f"<p style='color:#60a5fa; font-size:0.85rem; margin-top:-10px;'>Showing {len(filtered_df)} records</p>", unsafe_allow_html=True)

                        # Render the specific user's logs
                        for _, row in filtered_df.iterrows():
                            action = row['action_type']
                            ts = row['timestamp']
                            log_email = row['email']
                            details = row['details']
                            prompt = row['prompt']
                            response = row['response']

                            # If it's an AI tool use, show the expander with prompt/response
                            if pd.notna(prompt) and str(prompt).strip() != "":
                                with st.expander(f"{log_email} | {action} | {ts}"):
                                    st.markdown("**Prompt / Input:**"); st.info(prompt)
                                    st.markdown("**AI Response:**"); st.success(response)
                            # If it's just a system log (login, profile change), show the clean card
                            else:
                                st.markdown(f"<div class='clean-card'><p class='card-title'>{log_email} - {action} <span style='float:right; color:#8e918f; font-size:0.8rem; font-weight:400;'>{ts}</span></p><p class='card-text'>{details}</p></div>", unsafe_allow_html=True)
                    else:
                        st.info("No system activity recorded yet.")
                except Exception as e:
                    st.error(f"Could not load activity logs: {e}")

            # 👇 MOVED: Data Exports is now exactly inside tab 5 👇
            with tab5:
                st.markdown("### Export System Data")
                st.markdown("<p style='color:#8e918f; font-size:0.9rem; margin-bottom: 20px;'>Download comprehensive database tables as CSV files for local Excel reporting.</p>", unsafe_allow_html=True)
                c1, c2, c3 = st.columns(3)
                try:
                    with c1:
                        users_csv = pd.read_sql("SELECT id, username, email, created_at, role, is_blocked FROM users", conn).to_csv(index=False)
                        st.download_button("Export Full User List", data=users_csv, file_name="users_export.csv", mime="text/csv", use_container_width=True)
                    with c2:
                        activity_csv = pd.read_sql("SELECT * FROM user_activity", conn).to_csv(index=False)
                        st.download_button("Export Activity Logs", data=activity_csv, file_name="activity_logs.csv", mime="text/csv", use_container_width=True)
                    with c3:
                        feed_csv = pd.read_sql("SELECT * FROM tool_feedback", conn).to_csv(index=False)
                        st.download_button("Export AI Tool Feedback", data=feed_csv, file_name="tool_feedback.csv", mime="text/csv", use_container_width=True)
                except Exception as e:
                    st.error("Error connecting to database for export.")

    else:
        if st.session_state.menu_option == "Dashboard": dashboard_page(st.session_state.username, chunks)

        elif st.session_state.menu_option == "Profile":
            st.header("User Profile & Settings")
            tab_id, tab_sec = st.tabs(["Profile Identity", "Security Settings"])

            with tab_id:
                st.markdown("### Profile Identity")
                st.markdown(f"<p style='color:#e3e3e3; font-size:1.4rem; font-weight:600; margin-bottom:0;'>{st.session_state.username}</p>", unsafe_allow_html=True)
                st.markdown(f"<p style='color:#60a5fa; font-size:0.95rem; margin-top:0; margin-bottom:20px;'>{st.session_state.email}</p>", unsafe_allow_html=True)

                user_avatar = get_user_avatar(st.session_state.email)
                if user_avatar:
                    st.markdown(f"<img src='data:image/png;base64,{user_avatar}' style='width:120px; height:120px; border-radius:50%; object-fit:cover; border:3px solid #3b82f6; margin-bottom:20px;'/>", unsafe_allow_html=True)
                else:
                    st.info("No avatar set. Upload one below!")

                new_avatar = st.file_uploader("Upload New Avatar (PNG/JPG - Max 5MB)", type=["png", "jpg", "jpeg"])
                if st.button("Save Avatar", type="primary"):
                    if new_avatar:
                        if new_avatar.size > 5 * 1024 * 1024:
                            st.error("Image file is too large! Please upload an image under 5MB.")
                        else:
                            base64_img = base64.b64encode(new_avatar.read()).decode('utf-8')
                            update_avatar(st.session_state.email, base64_img)
                            log_activity(st.session_state.email, "Profile", "Updated Profile Avatar.")
                            st.success("Avatar updated successfully!")
                            time.sleep(1)
                            st.rerun()
                    else:
                        st.error("Please select an image file first.")

            with tab_sec:
                col1, col2 = st.columns(2)
                with col1:
                    st.markdown("### Update Email Address")
                    # 👇 FIXED 3: Email OTP Verification Flow 👇
                    if not st.session_state.get("email_update_otp_sent"):
                        with st.form("update_email_form"):
                            new_email = st.text_input("New Email Address", placeholder="new@example.com")
                            curr_pass_email = st.text_input("Current Password", type="password", placeholder="Verify your identity")
                            if st.form_submit_button("Send Verification OTP"):
                                if not new_email or not curr_pass_email:
                                    st.error("All fields are required.")
                                elif not valid_email(new_email):
                                    st.error("Invalid email format.")
                                elif check_user_exists(new_email):
                                    st.error("Email already in use by another account.")
                                else:
                                    auth_res, _ = authenticate_user(st.session_state.email, curr_pass_email)
                                    if auth_res:
                                        otp = generate_otp()
                                        save_otp(new_email, otp)
                                        success, msg = send_otp_email(new_email, otp)
                                        if success:
                                            st.session_state.email_update_otp_sent = True
                                            st.session_state.pending_new_email = new_email
                                            st.rerun()
                                        else:
                                            st.error(f"Failed to send OTP: {msg}")
                                    else:
                                        st.error("Incorrect password.")
                    else:
                        st.info(f"An OTP has been sent to **{st.session_state.pending_new_email}**")
                        with st.form("verify_email_otp_form"):
                            otp_input = st.text_input("Enter 6-digit OTP", max_chars=6)
                            if st.form_submit_button("Verify & Update Email", type="primary"):
                                if verify_otp(st.session_state.pending_new_email, otp_input):
                                    update_user_email_db(st.session_state.email, st.session_state.pending_new_email)
                                    log_activity(st.session_state.pending_new_email, "Security", "Email address updated.")
                                    st.success("Email updated successfully! Logging you out...")
                                    time.sleep(1.5)
                                    for key in list(st.session_state.keys()):
                                        del st.session_state[key]
                                    st.rerun()
                                else:
                                    st.error("Invalid or expired OTP.")
                        if st.button("Cancel / Back", use_container_width=True):
                            st.session_state.email_update_otp_sent = False
                            st.session_state.pending_new_email = None
                            st.rerun()

                with col2:
                    st.markdown("### Change Password")
                    old_pass = st.text_input("Current Password", type="password", key="cp_old")
                    new_pass = st.text_input("New Password", type="password", key="cp_new")

                    is_len = len(new_pass) >= 6
                    is_upper = bool(re.search(r"[A-Z]", new_pass))
                    is_num = bool(re.search(r"\d", new_pass))
                    is_spec = bool(re.search(r"[^A-Za-z0-9]", new_pass))

                    if new_pass:
                        st.markdown(f"<p style='font-size:0.75rem; margin-top:-10px; margin-bottom:10px; color:#c4c7c5;'>" +
                                    (f"<span style='color:#81c995;'>✅ 6+ chars</span>" if is_len else "❌ 6+ chars") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Uppercase</span>" if is_upper else "❌ Uppercase") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Number</span>" if is_num else "❌ Number") + " &nbsp;|&nbsp; " +
                                    (f"<span style='color:#81c995;'>✅ Special</span>" if is_spec else "❌ Special") +
                                    "</p>", unsafe_allow_html=True)

                    confirm_pass = st.text_input("Confirm New Password", type="password", key="cp_conf")

                    if st.button("Update Password", type="primary", use_container_width=True):
                        if not old_pass or not new_pass or not confirm_pass:
                            st.error("All fields required.")
                        elif new_pass != confirm_pass:
                            st.error("New passwords do not match.")
                        elif not (is_len and is_upper and is_num and is_spec):
                            st.error("Password is too weak. Please meet all the security requirements.")
                        else:
                            auth_res, _ = authenticate_user(st.session_state.email, old_pass)
                            if auth_res:
                                if check_password_reused(st.session_state.email, new_pass):
                                    st.error("Cannot reuse a recent password.")
                                else:
                                    update_password(st.session_state.email, new_pass)
                                    st.success("Password updated successfully! Logging you out...")
                                    time.sleep(1.5)
                                    for key in list(st.session_state.keys()): del st.session_state[key]
                                    st.rerun()
                            else:
                                st.error("Incorrect current password.")

        elif st.session_state.menu_option == "History":
            st.header("My Activity History")
            history = get_user_history(st.session_state.email)
            if not history: st.info("No activity recorded yet.")
            else:
                for action, details, prompt, response, ts in history:
                    if prompt or response:
                        with st.expander(f"{action} | {ts}"):
                            st.markdown("**Your Prompt / Input:**"); st.info(prompt)
                            st.markdown("**AI Response:**"); st.success(response)
                    else: st.markdown(f"<div class='clean-card'><p class='card-title'>{action} <span style='float:right; color:#8e918f; font-size:0.8rem; font-weight:400;'>{ts}</span></p><p class='card-text'>{details}</p></div>", unsafe_allow_html=True)

        elif st.session_state.menu_option == "Feedback":
            st.header("Submit General Platform Feedback")
            st.markdown("<p style='color:#c4c7c5;'>Help us improve PolicyNav. Your feedback is sent directly to our administration team.</p>", unsafe_allow_html=True)
            with st.form("feedback_form"):
                rating = st.slider("Rate your experience (1 = Poor, 5 = Excellent)", 1, 5, 5)
                comments = st.text_area("Any suggestions, feature requests, or bugs to report?", height=150)
                if st.form_submit_button("Submit Feedback", type="primary"):
                    if not comments.strip(): st.error("Please write a comment or suggestion before submitting.")
                    else: submit_feedback(st.session_state.email, rating, comments); st.success("Thank you. Your feedback has been recorded.")

        elif st.session_state.menu_option == "Chat":
            st.header("AI Policy Assistant")
            q = st.text_input("Ask a question about government policies:")
            col1, col2, _ = st.columns([2, 3, 5])
            with col1: submit_btn = st.button("Ask AI", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="chat_lang", label_visibility="collapsed")

            if submit_btn and q:
                with st.spinner("Searching..."):
                    eng_q = translate_fast(q, target_lang, "English", t_tokenizer, t_model)
                    context = "\n\n".join(search_policy(eng_q, index, chunks, embed_model))
                    prompt = (
                        "You are a policy expert. Answer the question thoroughly using the context provided. "
                        "Include: what the scheme is, who is eligible, key benefits, and important amounts or dates. "
                        "Write at least 4-6 sentences. Cite the document name.\n\n"
                        f"Context:\n{context}\n\nQuestion:\n{eng_q}\n\nDetailed Answer:")
                    inputs = s_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(s_model.device)
                    outputs = s_model.generate(**inputs, max_new_tokens=400, min_length=80, num_beams=4, length_penalty=2.0, no_repeat_ngram_size=4, repetition_penalty=2.5, early_stopping=True)
                    eng_ans = s_tokenizer.decode(outputs[0], skip_special_tokens=True)
                    eng_ans = " ".join(eng_ans.split())
                    final_ans = translate_fast(eng_ans, 'English', target_lang, t_tokenizer, t_model)
                    final_ans = " ".join(final_ans.split())
                    st.session_state.chat_q = q
                    st.session_state.chat_ans = final_ans
                    st.session_state.chat_rated = False
                log_activity(st.session_state.email, "Chat", f"Queried: '{q}'", prompt=q, response=final_ans)

            if st.session_state.get("chat_ans"):
                st.markdown(f"<div class='clean-card' style='padding:1.8rem 2rem;'><p class='card-title' style='font-size:0.8rem;margin-bottom:0.8rem;'>AI Answer</p><p style='color:#e3e3e3;font-size:1rem;line-height:1.85;white-space:pre-wrap;margin:0;'>{st.session_state.chat_ans}</p></div>", unsafe_allow_html=True)
                if not st.session_state.get("chat_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("chat_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this model by rating the specific response above.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Output Accuracy:", ["Excellent", "Good", "Average", "Poor", "Incorrect / Hallucination"], horizontal=True)
                        t_comment = st.text_input("Optional Notes (e.g., 'Model hallucinated a date' or 'Perfect translation')")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "AI Policy Assistant", st.session_state.chat_q, st.session_state.chat_ans, t_rating, t_comment)
                            st.session_state.chat_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this generation.")

        elif st.session_state.menu_option == "Summarization":
            st.header("AI Policy Summarizer")
            st.markdown("<p style='color:#8e918f;'>Paste a policy document excerpt or upload a PDF. The AI will produce 3 concise bullet-point summaries in your chosen language.</p>", unsafe_allow_html=True)

            tab_paste, tab_pdf = st.tabs(["Paste Text", "Upload PDF"])
            text_to_summarize = ""
            with tab_paste:
                raw_sum = st.text_area("Paste policy text here:", height=220, key="sum_raw")
                if raw_sum: text_to_summarize = raw_sum
            with tab_pdf:
                sum_pdf = st.file_uploader("Upload PDF", type=["pdf"], key="sum_pdf")
                if sum_pdf:
                    try:
                        reader = PyPDF2.PdfReader(sum_pdf)
                        text_to_summarize = "\n".join(p.extract_text() or "" for p in reader.pages)
                        st.success(f"Extracted ~{len(text_to_summarize.split())} words from {sum_pdf.name}")
                    except Exception as e:
                        st.error(f"PDF read error: {e}")

            col1, col2, _ = st.columns([2, 3, 5])
            with col1: sum_btn = st.button("Generate Summary", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="sum_lang", label_visibility="collapsed")

            if sum_btn:
                if len(text_to_summarize.strip()) < 50: st.error("Enter at least 50 characters.")
                else:
                    with st.spinner("Generating summary..."):
                        prompt = (
                            "Summarize the following policy document in detail. "
                            "Produce exactly 5 bullet points covering: "
                            "(1) What the scheme is and its objective, "
                            "(2) Who is eligible, "
                            "(3) Key financial benefits or amounts, "
                            "(4) How to apply or avail the scheme, "
                            "(5) Important dates, targets, or recent updates.\n\n"
                            f"Policy Text:\n{text_to_summarize[:3000]}\n\nDetailed Summary:")
                        inputs = s_tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to(s_model.device)
                        outputs = s_model.generate(**inputs, max_new_tokens=512, min_length=120, num_beams=4, length_penalty=2.0, no_repeat_ngram_size=4, repetition_penalty=2.5, early_stopping=True)
                        eng_summary = s_tokenizer.decode(outputs[0], skip_special_tokens=True)
                        eng_summary = " ".join(eng_summary.split())
                        final_summary = translate_fast(eng_summary, "English", target_lang, t_tokenizer, t_model)
                        final_summary = " ".join(final_summary.split())

                        st.session_state.sum_q = text_to_summarize[:300]
                        st.session_state.sum_ans = final_summary
                        st.session_state.sum_rated = False
                    log_activity(st.session_state.email, "Summarization", f"Summary in {target_lang}", prompt=text_to_summarize[:300], response=final_summary)

            if st.session_state.get("sum_ans"):
                st.markdown(f"<div class='clean-card' style='padding:1.8rem 2rem;'><p class='card-title' style='font-size:0.8rem;margin-bottom:0.8rem;'>AI Summary</p><p style='color:#e3e3e3;font-size:1rem;line-height:1.85;white-space:pre-wrap;margin:0;'>{st.session_state.sum_ans}</p></div>", unsafe_allow_html=True)
                if not st.session_state.get("sum_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("sum_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this model by rating the specific response above.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Output Accuracy:", ["Excellent", "Good", "Average", "Poor", "Missed Context / Too Long"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "AI Policy Summarizer", st.session_state.sum_q, st.session_state.sum_ans, t_rating, t_comment)
                            st.session_state.sum_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this generation.")

        elif st.session_state.menu_option == "Graph":
            st.header("Entity Knowledge Graph")
            num_chunks = st.slider("Number of document chunks to analyse", 10, 60, 15, 5, key="kg_slider")
            if st.button("Generate Graph", type="primary", use_container_width=True):
                with st.spinner("Extracting entities and building graph..."):
                    try:
                        graph_path = generate_knowledge_graph(chunks[:num_chunks], nlp)
                        st.session_state.kg_graph_path = graph_path
                        log_activity(st.session_state.email, "Knowledge Graph", "Rendered graph")
                        st.success("Graph generated successfully!")
                    except Exception as e:
                        st.error(f"Graph generation failed: {e}")
            render_path = st.session_state.get("kg_graph_path")
            if render_path and os.path.exists(render_path):
                try:
                    with open(render_path, "r", encoding="utf-8") as f:
                        html_content = f.read()
                    if html_content.strip():
                        components.html(html_content, height=500, scrolling=True)
                    else:
                        st.warning("Graph file is empty. Click Generate Graph to rebuild.")
                except Exception as e:
                    st.error(f"Could not render graph: {e}")
            else:
                st.info("Click Generate Graph to build the knowledge graph from your policy documents.")

        elif st.session_state.menu_option == "Web":
            st.header("Global Web Search")
            query = st.text_input("Enter research query:")

            if st.button("Search Web", type="primary") and query:
                st.session_state.web_rated = False
                st.session_state.web_ans = None
                with st.spinner("Retrieving..."):
                    results = global_web_research(query)
                    formatted_results = ""
                    for r in results:
                        st.markdown(f"<div class='clean-card'><p class='card-title'>{r['title']}</p><p class='card-text'>{r['body']}</p><a href='{r['href']}' style='color:#a8c7fa; font-size:0.85rem; text-decoration:none;'>Read source document</a></div>", unsafe_allow_html=True)
                        formatted_results += f"**{r['title']}**\n{r['body']}\n\n"

                    st.session_state.web_q = query
                    st.session_state.web_ans = formatted_results
                    st.session_state.web_rated = False
                log_activity(st.session_state.email, "Web Research", f"Researched: '{query}'", prompt=query, response=formatted_results)

            if st.session_state.get("web_ans"):
                if not st.session_state.get("web_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("web_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this search tool by rating the results.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Search Relevance:", ["Excellent", "Good", "Average", "Poor", "Irrelevant Results"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Global Web Search", st.session_state.web_q, st.session_state.web_ans, t_rating, t_comment)
                            st.session_state.web_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this search.")

        elif st.session_state.menu_option == "Readability":
            st.header("Readability Analyzer")
            tab1, tab2 = st.tabs(["Paste Text", "Upload File"]); text_input = ""
            with tab1:
                raw_text = st.text_area("Enter text below", height=180, key="read_txt")
                if raw_text: text_input = raw_text
            with tab2:
                uploaded_file = st.file_uploader("Upload file", type=["txt", "pdf"])
                if uploaded_file:
                    if uploaded_file.type == "application/pdf": text_input = "".join([page.extract_text() + "\n" for page in PyPDF2.PdfReader(uploaded_file).pages])
                    else: text_input = uploaded_file.read().decode("utf-8")

            if st.button("Analyze Text", type="primary"):
                if len(text_input.strip()) < 50: st.error("Text too short.")
                else:
                    with st.spinner("Calculating..."):
                        scores = readability.ReadabilityAnalyzer(text_input).get_all_metrics()

                        st.session_state.read_q = text_input[:200] + "..."
                        st.session_state.read_ans = f"Flesch Ease: {scores['Flesch Reading Ease']:.1f} | Grade: {scores['Flesch-Kincaid Grade']:.1f}"
                        st.session_state.read_scores = scores
                        st.session_state.read_rated = False
                    log_activity(st.session_state.email, "Readability", "Analyzed metrics", prompt=st.session_state.read_q, response=st.session_state.read_ans)

            if st.session_state.get("read_scores"):
                sc = st.session_state.read_scores
                fk = sc["Flesch-Kincaid Grade"]
                st.markdown("<br>", unsafe_allow_html=True)
                if fk <= 6:   lvl_icon,lvl_title,lvl_desc = "📗","Easy","Suitable for general public"
                elif fk <= 10: lvl_icon,lvl_title,lvl_desc = "📘","Moderate","High-school reading level"
                elif fk <= 14: lvl_icon,lvl_title,lvl_desc = "📙","Advanced","University level"
                else:          lvl_icon,lvl_title,lvl_desc = "📕","Expert","Specialist / legal audience"
                st.markdown(
                    f"<div style='background:#1e1f20;border:1px solid #444746;border-radius:12px;"
                    f"padding:1.5rem 2rem;display:flex;align-items:center;gap:1.5rem;margin-bottom:1.5rem;'>"
                    f"<span style='font-size:2.5rem;'>{lvl_icon}</span>"
                    f"<div><p style='font-size:1.2rem;font-weight:600;color:#e3e3e3;margin:0;'>{lvl_title}</p>"
                    f"<p style='font-size:.85rem;color:#8e918f;margin:0;'>{lvl_desc}</p></div>"
                    f"<div style='margin-left:auto;text-align:center;'>"
                    f"<p style='font-size:2.4rem;font-weight:600;color:#e3e3e3;margin:0;line-height:1;'>{fk:.1f}</p>"
                    f"<p style='font-size:.72rem;color:#8e918f;text-transform:uppercase;'>Grade Level</p></div></div>",
                    unsafe_allow_html=True)
                c1, c2, c3 = st.columns(3)
                with c1: st.markdown(metric_card("Flesch Reading Ease","0=hard to read, 100=easy to read",sc["Flesch Reading Ease"],sc["Flesch Reading Ease"],"#a8c7fa"), unsafe_allow_html=True)
                with c2: st.markdown(metric_card("Gunning Fog Index","Years of education needed to understand",sc["Gunning Fog"],(sc["Gunning Fog"]/20)*100,"#f59e0b"), unsafe_allow_html=True)
                with c3: st.markdown(metric_card("SMOG Index","Grade level required",sc["SMOG Index"],(sc["SMOG Index"]/20)*100,"#f28b82"), unsafe_allow_html=True)
                an2 = readability.ReadabilityAnalyzer(st.session_state.get("read_q",""))
                st.markdown(
                    "<div style='display:flex;gap:.8rem;margin-top:1rem;flex-wrap:wrap;'>"
                    + "".join([
                        f"<div style='background:#1e1f20;border:1px solid #444746;border-radius:10px;"
                        f"padding:.8rem 1.2rem;flex:1;min-width:100px;text-align:center;'>"
                        f"<span style='font-size:1.4rem;font-weight:500;color:#e3e3e3;display:block;'>{v}</span>"
                        f"<span style='font-size:.7rem;color:#8e918f;text-transform:uppercase;"
                        f"font-weight:500;letter-spacing:.05em;'>{l}</span></div>"
                        for v,l in [(an2.num_words,"Words"),(an2.num_sentences,"Sentences"),(an2.char_count,"Characters")]
                    ]) + "</div>",
                    unsafe_allow_html=True)

                if not st.session_state.get("read_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("read_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune the readability analyzer.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Accuracy of Math/Analysis:", ["Excellent", "Good", "Average", "Poor", "Failed to Parse Text"], horizontal=True)
                        t_comment = st.text_input("Optional Notes")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Readability Analyzer", st.session_state.read_q, st.session_state.read_ans, t_rating, t_comment)
                            st.session_state.read_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this analysis.")

        elif st.session_state.menu_option == "Translation":
            st.header("Language Translator")
            source_lang = st.selectbox("Detect Source Language:", list(LANG_CODES.keys()), index=0)
            text_to_translate = st.text_area("Paste text to translate:", height=200)
            col1, col2, _ = st.columns([2, 3, 5])
            with col1: trans_btn = st.button("Translate Text", type="primary", use_container_width=True)
            with col2: target_lang = st.selectbox("Output Language", list(LANG_CODES.keys()), key="trans_lang", label_visibility="collapsed")

            if trans_btn:
                if text_to_translate.strip() == "": st.error("Please paste text.")
                else:
                    with st.spinner("Translating..."):
                        result = translate_fast(text_to_translate, source_lang, target_lang, t_tokenizer, t_model)

                        st.session_state.trans_q = text_to_translate
                        st.session_state.trans_ans = result
                        st.session_state.trans_rated = False
                    log_activity(st.session_state.email, "Translation", f"To {target_lang}", prompt=text_to_translate, response=result)

            if st.session_state.get("trans_ans"):
                st.markdown(f"<div class='clean-card'><p class='card-title'>Translation</p><p class='card-text'>{st.session_state.trans_ans}</p></div>", unsafe_allow_html=True)

                if not st.session_state.get("trans_rated"):
                    st.markdown("<br><p style='color:#a8c7fa; font-size:0.9rem; font-weight:600; text-transform:uppercase; letter-spacing:1px;'>Model Performance Feedback</p>", unsafe_allow_html=True)
                    with st.form("trans_feed_form", clear_on_submit=True):
                        st.markdown("<p style='color:#c4c7c5; font-size:0.85rem; margin-bottom:10px;'>Help the admin team tune this translation model.</p>", unsafe_allow_html=True)
                        t_rating = st.radio("Translation Accuracy:", ["Excellent", "Good", "Average", "Poor Grammar", "Incorrect Language / Output"], horizontal=True)
                        t_comment = st.text_input("Optional Notes (e.g., 'Grammar was messy')")
                        if st.form_submit_button("Submit Data to Admin", type="primary"):
                            submit_tool_feedback(st.session_state.email, "Language Translator", st.session_state.trans_q, st.session_state.trans_ans, t_rating, t_comment)
                            st.session_state.trans_rated = True
                            st.rerun()
                else:
                    st.success("Performance data successfully logged for this translation.")

else:
    if st.session_state.page == "signup": signup()
    elif st.session_state.page == "forgot": forgot_password()
    else: login()

In [ ]:
from google.colab import userdata
import os

SECRETS = {
    'JWT_SECRET':         userdata.get('JWT_SECRET_KEY'),
    'EMAIL_ID':           userdata.get('EMAIL_ID'),
    'EMAIL_APP_PASSWORD': userdata.get('EMAIL_APP_PASSWORD'),
    'ADMIN_EMAIL':        userdata.get('ADMIN_EMAIL_ID'),
    'ADMIN_PASSWORD':     userdata.get('ADMIN_PASSWORD'),
    'NGROK_AUTHTOKEN':    userdata.get('NGROK_AUTHTOKEN'),
}
print('Secrets loaded:', [k for k,v in SECRETS.items() if v])

In [ ]:
from pyngrok import ngrok, conf
import subprocess, time, os

env = os.environ.copy()
env['JWT_SECRET']         = SECRETS['JWT_SECRET']         or ''
env['EMAIL_ID']           = SECRETS['EMAIL_ID']           or ''
env['EMAIL_APP_PASSWORD'] = SECRETS['EMAIL_APP_PASSWORD'] or ''
env['ADMIN_EMAIL']        = SECRETS['ADMIN_EMAIL']        or ''
env['ADMIN_PASSWORD']     = SECRETS['ADMIN_PASSWORD']     or ''
env['APP_DIR']            = os.environ.get('APP_DIR','/content/drive/MyDrive/PolicyNav')

conf.get_default().auth_token = SECRETS['NGROK_AUTHTOKEN']
ngrok.kill(); time.sleep(2)
subprocess.run(['pkill','-f','streamlit'], capture_output=True)
time.sleep(2)

subprocess.Popen(
    ['streamlit','run','app.py','--server.port=8501',
     '--server.headless=true','--browser.gatherUsageStats=false'],
    env=env,
    stdout=open('streamlit.log','w'),
    stderr=subprocess.STDOUT)
time.sleep(6)

public_url = ngrok.connect(8501)
print('PolicyNav is LIVE!')
print('URL:', public_url)